# UMAP: Supervised ViT-B/16 CLS Embedding Space

Visualises the frozen ViT-B/16 `[CLS]` embedding space used by the supervised
defect sweep.  All embeddings were cached during the sweep run (no GPU required here).

**Defect labels** are reconstructed from `LSWMD.pkl` by replaying the same
filtering logic as `load_defects_from_pickle` in the sweep script - without
loading images - so the label array aligns row-for-row with `all_defect_emb.npy`.

**Two plots:**
1. Coloured by role: train normals (background grey), test normals (blue),
   seen defects (teal), holdout Scratch (red), holdout Loc (orange).
2. Coloured by defect type: each of the 8 types gets its own colour,
   holdout types (Scratch, Loc) are marked with ★ in the legend.

UMAP is fit on a 5 000-point subsample of train normals only;
all other points are projected via `transform`.

## Imports and Paths

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import umap

# -- Repo root --------------------------------------------------------------
cwd = Path.cwd().resolve()
REPO_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "src" / "wafer_defect").exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise RuntimeError("Could not find repo root containing src/wafer_defect")

SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from wafer_defect.data.legacy_pickle import read_legacy_pickle, unwrap_legacy_value

# -- Experiment paths -------------------------------------------------------
NOTEBOOK_DIR   = REPO_ROOT / "experiments/anomaly_detection_defect/supervised_sweep/vit_b16/x224/main/umap"
MAIN_DIR       = NOTEBOOK_DIR.parent
ARTIFACT_DIR   = MAIN_DIR / "artifacts" / "supervised_vit_sweep"
EMBEDDING_DIR  = ARTIFACT_DIR / "embeddings"
RESULTS_DIR    = ARTIFACT_DIR / "results"
TRAIN_CONFIG   = MAIN_DIR / "train_config.toml"

UMAP_DIR = NOTEBOOK_DIR / "artifacts"
UMAP_DIR.mkdir(parents=True, exist_ok=True)

RAW_PICKLE = REPO_ROOT / "data" / "raw" / "LSWMD.pkl"

print(f"Repo root     : {REPO_ROOT}")
print(f"Embedding dir : {EMBEDDING_DIR} (exists={EMBEDDING_DIR.exists()})")
print(f"Raw pickle    : {RAW_PICKLE} (exists={RAW_PICKLE.exists()})")
print(f"Output dir    : {UMAP_DIR}")

## Load Sweep Config

In [ ]:
sweep_summary = json.loads((RESULTS_DIR / "sweep_summary.json").read_text(encoding="utf-8"))
cfg = sweep_summary["config"]

HOLDOUT_TYPES = set(cfg["holdout_defect_types"])   # {"Scratch", "Loc"}
SEED          = 42                                   # matches train_config.toml
IMAGE_SIZE    = 224                                  # matches train_config.toml [data] image_size

print(f"Holdout types : {HOLDOUT_TYPES}")
print(f"Seed          : {SEED}")
print(f"Image size    : {IMAGE_SIZE}")

## Reconstruct Defect Type Labels from LSWMD.pkl

The sweep script built `all_defect_emb.npy` by iterating `LSWMD.pkl` in
`iterrows()` order and appending a record for every row where `trianTestLabel`
is set **and** `failureType` is not empty / `'none'`.  We replay the same loop
here - skipping the image loading - to get a label for each row of the embedding.

In [ ]:
LABELS_CACHE = UMAP_DIR / "defect_labels.csv"

if LABELS_CACHE.exists():
    defect_labels_df = pd.read_csv(LABELS_CACHE)
    print(f"Loaded cached labels: {len(defect_labels_df)} defects")
else:
    if not RAW_PICKLE.exists():
        raise FileNotFoundError(
            f"LSWMD.pkl not found at {RAW_PICKLE}.\n"
            "Place LSWMD.pkl in data/raw/ and re-run this cell."
        )

    print(f"Reading {RAW_PICKLE} (this takes ~30-60 s) ...", flush=True)
    raw_df = read_legacy_pickle(RAW_PICKLE)

    records = []
    for _, row in raw_df.iterrows():
        train_label = unwrap_legacy_value(row.get("trianTestLabel", ""))
        if not train_label:
            continue  # unlabeled
        failure = unwrap_legacy_value(row.get("failureType", "")).strip()
        if not failure or failure.lower() in {"none", ""}:
            continue  # normal
        records.append({"emb_idx": len(records), "defect_type": failure})

    defect_labels_df = pd.DataFrame(records)
    defect_labels_df.to_csv(LABELS_CACHE, index=False)
    print(f"Saved {len(defect_labels_df)} defect labels to {LABELS_CACHE}")

print("\nClass counts:")
print(defect_labels_df["defect_type"].value_counts().to_string())

## Load Cached Embeddings

In [ ]:
print("Loading embeddings ...", flush=True)
all_defect_emb   = np.load(EMBEDDING_DIR / "all_defect_emb.npy")    # (25519, 768)
train_normal_emb = np.load(EMBEDDING_DIR / "train_normal_emb.npy")  # (40000, 768)
test_normal_emb  = np.load(EMBEDDING_DIR / "test_normal_emb.npy")   # (5000,  768)

assert len(all_defect_emb) == len(defect_labels_df), (
    f"Label count ({len(defect_labels_df)}) != embedding count ({len(all_defect_emb)}).\n"
    "The LSWMD.pkl filtering produced a different number of defects than the sweep script.\n"
    "Delete umap/artifacts/defect_labels.csv and re-run to rebuild labels."
)

defect_types = defect_labels_df["defect_type"].tolist()

print(f"all_defect_emb   : {all_defect_emb.shape}")
print(f"train_normal_emb : {train_normal_emb.shape}")
print(f"test_normal_emb  : {test_normal_emb.shape}")

## Fit UMAP on Train Normals, Project All Points

UMAP is fit on a 5 000-point random subsample of train normals (same pattern as the
classifier UMAP).  All other points - test normals and all defect types - are
projected via `transform`, so the manifold is defined entirely by the normal distribution.

In [ ]:
UMAP_COORDS_CSV = UMAP_DIR / "umap_coords.csv"

N_TRAIN_SAMPLE = 5_000   # train normals used to fit the manifold
UMAP_N_NEIGHBORS = 30
UMAP_MIN_DIST    = 0.10
UMAP_METRIC      = "cosine"   # matches the PatchCore scoring metric for ViT-B/16

if UMAP_COORDS_CSV.exists():
    umap_df = pd.read_csv(UMAP_COORDS_CSV)
    print(f"Loaded cached UMAP coords: {len(umap_df)} points")
else:
    rng = np.random.default_rng(SEED)
    train_idx = rng.choice(len(train_normal_emb), size=N_TRAIN_SAMPLE, replace=False)
    train_sample = train_normal_emb[train_idx]

    print(f"Fitting UMAP on {N_TRAIN_SAMPLE} train normals ...", flush=True)
    reducer = umap.UMAP(
        n_neighbors=UMAP_N_NEIGHBORS,
        min_dist=UMAP_MIN_DIST,
        metric=UMAP_METRIC,
        random_state=SEED,
        verbose=True,
    )
    reducer.fit(train_sample)

    # Project all point groups
    print("Projecting train normal sample ...", flush=True)
    train_coords = reducer.transform(train_sample)

    print("Projecting test normals ...", flush=True)
    test_coords = reducer.transform(test_normal_emb)

    print("Projecting defects ...", flush=True)
    defect_coords = reducer.transform(all_defect_emb)

    # Build combined DataFrame
    rows = []
    for u1, u2 in train_coords:
        rows.append({"umap_1": u1, "umap_2": u2, "group": "train_normal", "defect_type": "normal"})
    for u1, u2 in test_coords:
        rows.append({"umap_1": u1, "umap_2": u2, "group": "test_normal",  "defect_type": "normal"})
    for (u1, u2), dtype in zip(defect_coords, defect_types):
        is_holdout = dtype in HOLDOUT_TYPES
        group = f"holdout_{dtype.lower()}" if is_holdout else "seen_defect"
        rows.append({"umap_1": u1, "umap_2": u2, "group": group, "defect_type": dtype})

    umap_df = pd.DataFrame(rows)
    umap_df.to_csv(UMAP_COORDS_CSV, index=False)
    print(f"Saved UMAP coords to {UMAP_COORDS_CSV}")

print("\nGroup counts:")
print(umap_df["group"].value_counts().to_string())

## Plot 1: Coloured by Role

Train normals (background grey), test normals (blue),
seen defects (teal), holdout Scratch (crimson), holdout Loc (orange).

In [ ]:
PLOT1_PATH = UMAP_DIR / "umap_by_role.png"

role_styles = {
    "train_normal":    dict(color="#d1d5db", s=4,  alpha=0.25, zorder=1, label="Train normal"),
    "test_normal":     dict(color="#4d908e", s=8,  alpha=0.40, zorder=2, label="Test normal"),
    "seen_defect":     dict(color="#277da1", s=14, alpha=0.65, zorder=3, label="Seen defect (6 classes)"),
    "holdout_scratch": dict(color="#e63946", s=22, alpha=0.85, zorder=5, label="Holdout - Scratch ★"),
    "holdout_loc":     dict(color="#9b5de5", s=22, alpha=0.85, zorder=4, label="Holdout - Loc ★"),
}

fig, ax = plt.subplots(figsize=(10, 7))

for group_name in ["train_normal", "test_normal", "seen_defect", "holdout_loc", "holdout_scratch"]:
    mask = umap_df["group"] == group_name
    sub = umap_df[mask]
    style = role_styles[group_name]
    ax.scatter(sub["umap_1"], sub["umap_2"], **style)

ax.set_title(
    "ViT-B/16 CLS embeddings - coloured by role\n"
    "UMAP fit on train normals only; all other points projected via transform.",
    fontsize=11,
)
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
fig.savefig(PLOT1_PATH, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)
print("Saved:", PLOT1_PATH)

## Plot 2: Coloured by Defect Type

Each of the 8 defect types gets its own colour.  Holdout types (Scratch, Loc)
are marked with ★ in the legend and plotted on top.  Normal wafers are light grey.

This plot reveals *why* holdout recall is morphology-dependent:
- **Loc** shares cluster morphology with Center / Edge-Loc - its embeddings
  should sit near those seen-class clusters, explaining partial recall.
- **Scratch** has a linear, low-contrast signature with no structural
  analogue in the seen classes - its embeddings should be isolated or
  intermingled with normals, explaining near-zero recall.

In [ ]:
PLOT2_PATH = UMAP_DIR / "umap_by_defect_type.png"

all_types = sorted(umap_df["defect_type"].unique())
seen_types_list = [t for t in all_types if t != "normal" and t not in HOLDOUT_TYPES]

tab10 = plt.get_cmap("tab10")
type_color = {}
for i, t in enumerate(seen_types_list):
    type_color[t] = tab10(i % 10)
type_color["Scratch"] = "#e63946"   # vivid red
type_color["Loc"]     = "#9b5de5"   # vivid purple (distinct from red)
type_color["normal"]  = "#d1d5db"   # light grey

fig, ax = plt.subplots(figsize=(10, 7))

# Normal wafers as faint background
norm_mask = umap_df["defect_type"] == "normal"
ax.scatter(
    umap_df.loc[norm_mask, "umap_1"],
    umap_df.loc[norm_mask, "umap_2"],
    s=4, alpha=0.20, color=type_color["normal"], zorder=1, label="Normal",
)

# Seen defect classes
for dtype in seen_types_list:
    mask = umap_df["defect_type"] == dtype
    sub = umap_df[mask]
    ax.scatter(
        sub["umap_1"], sub["umap_2"],
        s=14, alpha=0.70, color=type_color[dtype], zorder=3, label=dtype,
    )

# Holdout classes on top with star marker
for dtype in sorted(HOLDOUT_TYPES):
    mask = umap_df["defect_type"] == dtype
    sub = umap_df[mask]
    ax.scatter(
        sub["umap_1"], sub["umap_2"],
        s=40, alpha=0.90, color=type_color[dtype],
        marker="*", zorder=5, label=f"{dtype} ★ (holdout)",
    )

ax.set_title(
    "ViT-B/16 CLS embeddings - coloured by defect type\n"
    "★ = holdout classes (Scratch, Loc) withheld from training at all fractions.",
    fontsize=11,
)
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
ax.legend(frameon=False, fontsize=9, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
fig.savefig(PLOT2_PATH, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)
print("Saved:", PLOT2_PATH)

## Summary

## Plot 3: Balanced Subsample - Report View

With 25 k defects in the full plot, large seen classes (Edge-Ring: 13k+) dominate
and obscure the relative positions of smaller classes.  This plot caps each defect
type at **200 points** and normals at **3 000 points**, giving equal visual weight
to every class.

The key things to read from this plot:
- **Scratch (red ★)** - if it lands inside or adjacent to the normal mass, the linear
  probe cannot separate it from normals regardless of how many seen-class examples
  are added.  That is the geometric reason for near-zero holdout recall.
- **Loc (purple ★)** - if it sits near Center / Edge-Loc clusters, the decision
  boundary learned from those seen classes partially extends to Loc, explaining the
  partial but plateauing recall (~0.69).

In [ ]:
PLOT3_PATH = UMAP_DIR / "umap_balanced_subsample.png"

N_PER_DEFECT_CLASS = 200
N_NORMALS          = 3_000

rng = np.random.default_rng(SEED)

# -- Subsample normals ------------------------------------------------------
norm_rows = umap_df[umap_df["defect_type"] == "normal"]
norm_sample = norm_rows.sample(n=min(N_NORMALS, len(norm_rows)), random_state=SEED)

# -- Subsample each defect type ---------------------------------------------
defect_rows = umap_df[umap_df["defect_type"] != "normal"]
parts = [norm_sample]
for dtype, grp in defect_rows.groupby("defect_type"):
    parts.append(grp.sample(n=min(N_PER_DEFECT_CLASS, len(grp)), random_state=SEED))
plot_df = pd.concat(parts, ignore_index=True)

# -- Plot -------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 7))

# Normal background
nm = plot_df["defect_type"] == "normal"
ax.scatter(
    plot_df.loc[nm, "umap_1"], plot_df.loc[nm, "umap_2"],
    s=6, alpha=0.25, color="#d1d5db", zorder=1, label="Normal",
)

# Seen defect classes (circle markers)
for dtype in seen_types_list:
    mask = plot_df["defect_type"] == dtype
    sub = plot_df[mask]
    if sub.empty:
        continue
    ax.scatter(
        sub["umap_1"], sub["umap_2"],
        s=20, alpha=0.75, color=type_color[dtype], zorder=3, label=dtype,
    )

# Holdout classes on top (star markers, larger)
for dtype in sorted(HOLDOUT_TYPES):
    mask = plot_df["defect_type"] == dtype
    sub = plot_df[mask]
    ax.scatter(
        sub["umap_1"], sub["umap_2"],
        s=60, alpha=0.90, color=type_color[dtype],
        marker="*", zorder=5, label=f"{dtype} ★ (holdout)",
    )

ax.set_title(
    f"ViT-B/16 CLS embeddings - balanced subsample ({N_PER_DEFECT_CLASS} per defect class, {N_NORMALS:,} normals)\n"
    "★ = holdout classes withheld from training at all sweep fractions.",
    fontsize=11,
)
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
ax.legend(frameon=False, fontsize=9, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
fig.savefig(PLOT3_PATH, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)
print("Saved:", PLOT3_PATH)

In [ ]:
print("Saved artefacts:")
for p in sorted(UMAP_DIR.glob("*")):
    print(f"  {p.relative_to(NOTEBOOK_DIR)}")